# benchmarking_atari_200m: improvements

In [ ]:
# Import the experiment config, the results loader and the analysis functions.
import importlib.util
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from config import EXPERIMENT

from analysis.curves import lifetime_average, return_curve
from analysis.stats import ATARI_REFERENCE_SCORES, human_normalize, mean_ci
from main import load_results

PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots" / "improvements"
BASELINE_CONFIG = Path("../atari_200m_deterministic/config.py")

In [ ]:
# Load Agent0's and Agent1's runs of every game with human and random scores, and
# the DQN runs of atari_200m_deterministic as the baseline.
component = EXPERIMENT.component("agent0_atari")
GAMES = sorted(
    game
    for game in component.sweep["ENV_HYPERS.GAME"]
    if game in ATARI_REFERENCE_SCORES
)

spec = importlib.util.spec_from_file_location("baseline_config", BASELINE_CONFIG)
baseline_config = importlib.util.module_from_spec(spec)
spec.loader.exec_module(baseline_config)
STORES = {
    "DQN": (baseline_config.EXPERIMENT, "dqn_atari"),
    "Agent0": (EXPERIMENT, "agent0_atari"),
    "Agent1": (EXPERIMENT, "agent1_atari"),
}

results = {
    (agent, game): load_results(experiment, name, where={"ENV_HYPERS.GAME": game})
    for agent, (experiment, name) in STORES.items()
    for game in GAMES
}

In [ ]:
# Human normalize each game's lifetime average return.
def human_normalized_auc(agent):
    return np.array([
        human_normalize(
            lifetime_average(
                return_curve(results[agent, game].reward, results[agent, game].done)
            ).mean(),
            game,
        )
        for game in GAMES
    ])


scores = {agent: human_normalized_auc(agent) for agent in STORES}

In [ ]:
# Style for all bar plots.
BAR_STYLE = {
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 20,
    "axes.labelsize": 20,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
}
SUMMARY_SIZE = (8, 6)
IMPROVEMENT_SIZE = (20, 7)
COLORS = {"DQN": "tab:blue", "Agent0": "tab:red", "Agent1": "tab:green"}
COMPARISONS = [("DQN", "Agent0"), ("DQN", "Agent1"), ("Agent0", "Agent1")]
CAPSIZE = 8
HNS_LABEL = "Human Normalized\nArea Under Curve"
COLORMAP = "berlin"


def title(game):
    return game.replace("_", " ").title()

In [ ]:
# Plot each agent's mean human normalized AUC across games, with a 95% bootstrap CI.
with plt.rc_context(BAR_STYLE):
    summary, ax = plt.subplots(figsize=SUMMARY_SIZE)
    for x, (agent, agent_scores) in enumerate(scores.items()):
        mean, low, high = mean_ci(agent_scores)
        ax.bar(x, mean, color=COLORS[agent])
        ax.errorbar(x, mean, yerr=[[mean - low], [high - mean]], color="grey",
                    capsize=CAPSIZE)
    ax.set_xticks(range(len(scores)), scores.keys())
    ax.set_ylabel(HNS_LABEL)
    summary.tight_layout()
figures = {"aggregate_bars": summary}
plt.show()

In [ ]:
# Plot each game's improvement for every comparison, sorted and colored by rank.
colors = plt.get_cmap(COLORMAP)(np.linspace(0, 1, len(GAMES)))
for before, after in COMPARISONS:
    improvement = scores[after] - scores[before]
    order = np.argsort(improvement)
    with plt.rc_context(BAR_STYLE):
        fig, ax = plt.subplots(figsize=IMPROVEMENT_SIZE)
        ax.bar(range(len(GAMES)), improvement[order], color=colors)
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_xticks(range(len(GAMES)), [title(GAMES[i]) for i in order], rotation=90)
        ax.set_xlim(-0.5, len(GAMES) - 0.5)
        ax.set_ylabel(f"Improvement in HNS\n{before}->{after}")
        fig.tight_layout()
    figures[f"improve_{before}_{after}".lower()] = fig
plt.show()

In [ ]:
# Save every plot as PDF and PNG in the plots directory.
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
for name, fig in figures.items():
    for ext in ("pdf", "png"):
        fig.savefig(PLOTS_DIR / f"{name}.{ext}", bbox_inches="tight")
print(f"saved {len(figures)} plot(s) to {PLOTS_DIR}")